# MUC-4 smoke fine-tune on a Colab T4

QLoRA fine-tune of `Qwen/Qwen3-0.6B` on the first 100 train documents, scoring the first 50 dev
documents every half epoch into one W&B run in `muc4-event-extraction` and pushing the adapter to
the Hub at every eval point. **Run all once: nothing here restarts the runtime.**

Needs the Colab Secrets `HF_TOKEN` (write scope) and `WANDB_API_KEY` (key icon in the left sidebar).
Set `REF` in the clone cell to run a specific commit, branch or tag.

**Train from the `!python -m train` cells only, never by calling `train.train()` in this kernel:**
Unsloth finishes the W&B run on a second training run in one Python process, abandoning the run
`train.py` configured. Each cell below starts its own process, so re-running one is safe.

In [ ]:
import sys

import torch

!nvidia-smi
print("torch", torch.__version__, "| python", sys.version)

In [ ]:
import importlib.util

# plain pip, not Unsloth's notebook recipe: Colab's torch already satisfies every pin
if importlib.util.find_spec("unsloth") is None:
    !pip install --quiet unsloth

In [ ]:
REF = "main"  # @param {type:"string"}
import os

%cd /content
if not os.path.isdir("fine-tuning-decoder"):
    !git clone --quiet https://github.com/fnl/fine-tuning-decoder.git
%cd /content/fine-tuning-decoder
!git fetch --quiet origin \
    && (git checkout --quiet --detach origin/$REF 2>/dev/null || git checkout --quiet --detach $REF) \
    && git log --oneline -1
!pip install --quiet -e .

In [ ]:
from importlib.metadata import version

import train  # noqa: F401  (the editable install put src/ on the path)

for package in ("fine-tuning-decoder", *train.VERSIONED):
    print(package, version(package))

In [ ]:
import os

from google.colab import userdata

for name in ("HF_TOKEN", "WANDB_API_KEY"):
    value = userdata.get(name)
    assert value, f"{name} is missing from Colab Secrets"
    os.environ[name] = value

In [ ]:
# smoke: 8 documents, a handful of steps; proves install, masking and the push before the real run
!python -m train --config configs/qwen3-0.6b-smoke.yaml --limit 8

In [ ]:
!python -m train --config configs/qwen3-0.6b-smoke.yaml

In [ ]:
# the adapter survived: load the *pushed* adapter onto a fresh base model and generate one document
import torch
import yaml
from datasets import load_dataset
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

from generate import generate, unsloth_engine

config = yaml.safe_load(open("configs/qwen3-0.6b-smoke.yaml"))
tokenizer = AutoTokenizer.from_pretrained(config["model"])
base = AutoModelForCausalLM.from_pretrained(config["model"], dtype=torch.float16, device_map="cuda")
model = PeftModel.from_pretrained(base, config["adapter"])
max_new_tokens = config["generation"]["max_new_tokens"]
engine = unsloth_engine(model, tokenizer, max_new_tokens=max_new_tokens, batch_size=1)
example = load_dataset(config["dataset"])["dev"][0]
[row] = generate([example], engine, tokenizer, max_input_tokens=config["max_seq_len"] - max_new_tokens)
print("gold:  ", example["messages"][2]["content"])
print("output:", row["output"], "(cut off)" if row["cut_off"] else "")